In [1]:
import rasterio

In [2]:
fp = r"r:\delovno\nejc\slrm4adaf\slrm\241726_4733686_rvt_SLRM_R20.tif"

In [6]:
fp = r"r:\delovno\nejc\slrm4adaf\slrm\243262_4769014_rvt_SLRM_R20.tif"

In [7]:
with rasterio.open(fp) as src:
    arr = src.read()
    profile = src.profile
    transform = src.transform
    crs = src.crs


In [8]:
profile

{'driver': 'GTiff', 'dtype': 'float32', 'nodata': 0.0, 'width': 512, 'height': 512, 'count': 1, 'crs': CRS.from_epsg(32634), 'transform': Affine(0.5, 0.0, 243262.5,
       0.0, -0.5, 4769270.0), 'blockysize': 4, 'tiled': False, 'compress': 'lzw', 'interleave': 'band'}

In [12]:
N = 8   # number of pixels to clip from each border

# Remove N pixels from edges
clipped = arr[:, N:-N, N:-N]

# Update transform: move origin by N pixels
new_transform = transform * transform.translation(N, N)

# Update profile with new width/height
profile.update({
    "height": clipped.shape[1],
    "width": clipped.shape[2],
    "transform": new_transform
})

# Write output
output_file = "_clipped.tif"

with rasterio.open(output_file, "w", **profile) as dst:
    dst.write(clipped)


In [ ]:
from osgeo import gdal

def build_vrt_overviews(
    vrt_path: str,
    levels = (2, 4, 8, 16, 32),
    resampling: str = "AVERAGE",
    compress: str = "LZW",
    bigtiff: str = "IF_SAFER",
    ovr_blocksize: int | None = 256,
):
    gdal.UseExceptions()

    gdal.SetConfigOption("COMPRESS_OVERVIEW", compress)
    gdal.SetConfigOption("BIGTIFF_OVERVIEW", bigtiff)
    if ovr_blocksize is not None:
        gdal.SetConfigOption("GDAL_TIFF_OVR_BLOCKSIZE", str(ovr_blocksize))

    ds = gdal.Open(vrt_path, gdal.GA_Update)
    if ds is None:
        raise RuntimeError(f"Could not open {vrt_path}")

    try:
        err = ds.BuildOverviews(resampling, list(levels))
    finally:
        ds = None

    if err != 0:
        raise RuntimeError(f"Error {err} while building overviews")
    print(f"Overviews built for {vrt_path} with levels {levels}")

# Example call:
build_vrt_overviews(r"/data/rasters/mosaic.vrt", resampling="AVERAGE")


In [1]:
import geopandas as gpd

In [2]:
gdf = gpd.read_file(r"r:\delovno\nejc\clipped_DEM_ws1_20251124_155108_seg\semantic_segmentation.gpkg")

In [3]:
gdf.head()

,label,prediction_path,roundness,area,geometry
0,custom,clipped_DEM_ws1_20251124_155108_seg\prediction...,0.929,300.0,"POLYGON ((292902.000 4750667.000, 292902.000 4..."
1,custom,clipped_DEM_ws1_20251124_155108_seg\prediction...,0.915,284.0,"POLYGON ((292866.500 4750708.500, 292867.000 4..."
2,custom,clipped_DEM_ws1_20251124_155108_seg\prediction...,0.839,50.0,"POLYGON ((292857.500 4750801.500, 292858.000 4..."
3,custom,clipped_DEM_ws1_20251124_155108_seg\prediction...,0.886,150.0,"POLYGON ((292584.500 4750807.000, 292584.500 4..."
4,custom,clipped_DEM_ws1_20251124_155108_seg\prediction...,0.929,413.0,"POLYGON ((292820.500 4750850.000, 292822.000 4..."


In [8]:
gdf.dissolve().explode(index_parts=False).reset_index()

,index,label,prediction_path,roundness,area,geometry
0,0,custom,clipped_DEM_ws1_20251124_155108_seg\prediction...,0.929,300.0,"POLYGON ((292988.000 4750513.500, 292988.000 4..."
1,0,custom,clipped_DEM_ws1_20251124_155108_seg\prediction...,0.929,300.0,"POLYGON ((292948.500 4750538.000, 292948.500 4..."
2,0,custom,clipped_DEM_ws1_20251124_155108_seg\prediction...,0.929,300.0,"POLYGON ((292935.500 4750569.500, 292935.500 4..."
3,0,custom,clipped_DEM_ws1_20251124_155108_seg\prediction...,0.929,300.0,"POLYGON ((292944.000 4750636.000, 292944.500 4..."
4,0,custom,clipped_DEM_ws1_20251124_155108_seg\prediction...,0.929,300.0,"POLYGON ((292904.000 4750665.500, 292904.000 4..."
5,0,custom,clipped_DEM_ws1_20251124_155108_seg\prediction...,0.929,300.0,"POLYGON ((292868.500 4750707.500, 292868.500 4..."
6,0,custom,clipped_DEM_ws1_20251124_155108_seg\prediction...,0.929,300.0,"POLYGON ((292924.000 4750770.500, 292923.000 4..."
7,0,custom,clipped_DEM_ws1_20251124_155108_seg\prediction...,0.929,300.0,"POLYGON ((292859.000 4750800.000, 292860.000 4..."
8,0,custom,clipped_DEM_ws1_20251124_155108_seg\prediction...,0.929,300.0,"POLYGON ((292587.000 4750805.500, 292588.000 4..."
9,0,custom,clipped_DEM_ws1_20251124_155108_seg\prediction...,0.929,300.0,"POLYGON ((292929.500 4750843.000, 292930.000 4..."


In [10]:
import rasterio
import geopandas as gpd
from shapely.geometry import box
from pathlib import Path

def build_extent_grid(input_folder, output_gpkg):
    input_folder = Path(input_folder)

    # Find all GeoTIFF files (supports .tif and .tiff)
    tif_files = sorted(input_folder.glob("*.tif")) + sorted(input_folder.glob("*.tiff"))

    if not tif_files:
        raise ValueError("No .tif/.tiff files found in the provided folder.")

    geoms = []
    names = []
    crs = None

    for f in tif_files:
        with rasterio.open(f) as src:
            # Use CRS from the first raster
            if crs is None:
                crs = src.crs
            elif src.crs != crs:
                raise ValueError(f"CRS mismatch in {f}: {src.crs} != {crs}")

            b = src.bounds
            geoms.append(box(b.left, b.bottom, b.right, b.top))
            names.append(f.name)

    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame(
        {"filename": names},
        geometry=geoms,
        crs=crs
    )

    # Save to GeoPackage
    gdf.to_file(output_gpkg, driver="GPKG")
    print(f"Saved grid to: {output_gpkg}")


# Example:
build_extent_grid(r"r:\delovno\nejc\stone_visualisations\BiH_ALS_2025_DMO_05m_slrm4inference", r"r:\delovno\nejc\stone_visualisations\BiH_ALS_2025_DMO_05m_slrm4inference.gpkg")


KeyboardInterrupt: 

In [ ]:
def assign_splits_to_grid(df_patches, split_gpkg, ds_crs):
    """
    Assign each patch to train / val / test based on polygons in split_gpkg.

    split_gpkg must contain polygons with attribute 'split' having values
    'validation' and/or 'test'. Everything else becomes 'train'.

    Tiles must be COMPLETELY inside validation/test polygons to be assigned there.
    Tiles intersecting a split polygon but not fully inside are dropped.
    """
    split_gdf = gpd.read_file(split_gpkg)
    # Reproject to raster CRS if needed
    if split_gdf.crs != ds_crs:
        split_gdf = split_gdf.to_crs(ds_crs)

    # Build union geometries for val and test
    val_geom = None
    test_geom = None

    if not split_gdf[split_gdf["split"].str.lower() == "validation"].empty:
        val_geom = split_gdf[split_gdf["split"].str.lower() == "validation"].dissolve().geometry.unary_union

    if not split_gdf[split_gdf["split"].str.lower() == "test"].empty:
        test_geom = split_gdf[split_gdf["split"].str.lower() == "test"].dissolve().geometry.unary_union

    def _tile_split(geom):
        # validation
        if val_geom is not None and geom.within(val_geom):
            return "train/val".split("/")[1] if False else "val"  # keep it simple: "val"
        # test
        if test_geom is not None and geom.within(test_geom):
            return "test"
        # If it intersects val/test but is not fully within -> discard
        if (val_geom is not None and geom.intersects(val_geom)) or \
           (test_geom is not None and geom.intersects(test_geom)):
            return None
        # everything else -> train
        return "train"

    df_patches["split"] = df_patches["geometry"].apply(_tile_split)

    # Drop tiles that are cut by split boundaries
    df_patches = df_patches[df_patches["split"].notna()].reset_index(drop=True)
    return df_patches